# 🧠 Prompt Engineering for AI Agents
## AI Agents Roadmap — Phase 1, Week 1–2

---

**Mức độ:** Intermediate → Advanced  
**Thời gian:** 3–5 ngày  
**Mục tiêu:** Nắm vững các kỹ thuật prompting để tối ưu LLMs cho AI Agents  

### 📋 Nội dung
1. Overview & Mental Model
2. Zero-shot / One-shot / Few-shot Prompting
3. Chain-of-Thought (CoT) & Variants
4. Role Prompting & System Prompt Design
5. Structured Prompting & Templates
6. Advanced Agent Techniques (ReAct, Meta-Prompting, Self-Critique)
7. Output Control & JSON Mode
8. Prompt Evaluation & A/B Testing
9. Model-specific Best Practices
10. Capstone: Production Prompt Library


## ⚙️ Setup & Dependencies

In [ ]:
# Cài đặt dependencies
!pip install anthropic openai jinja2 pydantic tiktoken -q

import os
import json
import asyncio
import hashlib
import statistics
from dataclasses import dataclass, field
from typing import Optional, Callable
from collections import Counter

import anthropic
from anthropic import Anthropic, AsyncAnthropic
from jinja2 import Template
from pydantic import BaseModel

# Setup API key
# Option 1: Set trực tiếp (chỉ dùng khi dev, KHÔNG commit lên git)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# Option 2 (Khuyên dùng): Load từ .env file
# from dotenv import load_dotenv; load_dotenv()

client = Anthropic()  # tự đọc ANTHROPIC_API_KEY từ env

# Helper function để gọi API nhanh
def ask(user_msg: str, system_msg: str = "", model: str = "claude-sonnet-4-6", max_tokens: int = 1024) -> str:
    """Wrapper đơn giản để gọi Anthropic API."""
    kwargs = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": user_msg}]
    }
    if system_msg:
        kwargs["system"] = system_msg
    response = client.messages.create(**kwargs)
    return response.content[0].text

print("✅ Setup hoàn tất!")

---
## 1. 🎯 Overview: Prompt Engineering là gì?

**Prompt Engineering** là nghệ thuật và khoa học thiết kế inputs (prompts) cho Large Language Models (LLMs) để nhận được outputs chất lượng cao, nhất quán và có thể dự đoán được.

### Tại sao quan trọng với AI Agents?
- Mỗi agent phụ thuộc vào prompt để **quyết định hành động**, gọi tool nào, và tạo ra output như thế nào
- Prompt viết tệ → agent hallucinate, bỏ qua bước quan trọng, hoặc tốn 10x token không cần thiết

### Mental Model: LLM là một Function

```
f(system_prompt, conversation_history, user_input) → output

Prompt Engineering = tối ưu hóa TẤT CẢ input parameters
                     để đạt được output quality tốt nhất
```

### Taxonomy of Techniques

| Nhóm | Kỹ thuật |
|------|----------|
| Basic | Zero-shot, One-shot, Few-shot |
| Reasoning Chains | CoT, Self-Consistency, Tree-of-Thought |
| Roles & Personas | Role Prompting, System Prompt Design |
| Structured Output | JSON mode, Schema constraints |
| Self-Improvement | Self-critique, Reflection |
| Agent-specific | ReAct, Tool-use, Planning prompts |

---
## 2. 📌 Zero-shot / One-shot / Few-shot Prompting

### 2.1 Zero-shot Prompting
Hỏi model thực hiện task **không có ví dụ nào**. Dựa hoàn toàn vào kiến thức pre-training.

In [ ]:
# ✅ Zero-shot: không có ví dụ
zero_shot_prompt = """
Classify the sentiment of the following review:

Review: "Terrible product, not worth the money, poor packaging"
Sentiment (positive/negative/neutral):
"""

result = ask(zero_shot_prompt)
print(f"Output: {result}")

# Nhận xét:
# ✓ Hoạt động tốt với các task đơn giản
# ✗ Format output không nhất quán
# ✗ Nhạy cảm với cách diễn đạt (phrasing)

### 2.2 Few-shot Prompting — Anatomy
Cung cấp 1+ demonstrations để dạy model **format và pattern** mong muốn.

In [ ]:
# ✅ Few-shot với format nhất quán
system = "You are a sentiment analysis expert for e-commerce platforms."

few_shot_prompt = """
# Example 1:
Review: "Fast delivery, sturdy packaging, product exactly as described"
Label: POSITIVE
Score: 0.95

# Example 2:
Review: "Quality is okay, slightly overpriced for what you get"
Label: NEUTRAL
Score: 0.40

# Example 3:
Review: "Item arrived defective, shop refused to help, very disappointed"
Label: NEGATIVE
Score: 0.05

# Analyze:
Review: "Nice design but battery life doesn't match the advertised specs"
Label:
Score:
"""

result = ask(few_shot_prompt, system_msg=system)
print(f"Output:\n{result}")

In [ ]:
# 🔬 EXPERIMENT: So sánh Zero-shot vs Few-shot
test_reviews = [
    "Amazing quality! But delivery took 2 weeks...",
    "Works fine I guess",
    "10/10 would buy again, exceeded expectations"
]

print("=" * 60)
print("COMPARISON: Zero-shot vs Few-shot")
print("=" * 60)

for review in test_reviews:
    print(f"\nReview: {review}")
    
    # Zero-shot
    zs_result = ask(f'Classify sentiment (positive/negative/neutral): "{review}"')
    print(f"  Zero-shot → {zs_result.strip()[:60]}")
    
    # Few-shot
    fs_prompt = few_shot_prompt.replace(
        '"Nice design but battery life doesn\'t match the advertised specs"',
        f'"{review}"'
    )
    fs_result = ask(fs_prompt, system_msg=system)
    print(f"  Few-shot  → {fs_result.strip()[:60]}")

### 2.3 Best Practices cho Few-shot

| Nguyên tắc | Giải thích |
|-----------|------------|
| **Quality over quantity** | 3–5 ví dụ tốt > 10 ví dụ tệ |
| **Diversify examples** | Cover các edge cases và biến thể quan trọng |
| **Consistent format** | Tất cả examples phải cùng structure và separators |
| **Order matters** | Ví dụ cuối cùng ảnh hưởng nhất (recency bias) |
| **Task first** | Đặt instructions trước examples |

> **💡 15 năm kinh nghiệm:** Dynamic few-shot selection — chọn examples giống input nhất qua embedding similarity — thường outperform static examples 15–20% accuracy.

In [ ]:
# 🛠️ PRACTICE: Dynamic Few-shot Selection (Simplified Demo)
# Trong production: dùng vector DB để embed + retrieve examples

EXAMPLE_POOL = [
    {"input": "Fast delivery, great quality", "label": "POSITIVE", "score": 0.92},
    {"input": "Arrived broken, waste of money", "label": "NEGATIVE", "score": 0.05},
    {"input": "Average product, nothing special", "label": "NEUTRAL", "score": 0.50},
    {"input": "Love the design but pricey", "label": "MIXED", "score": 0.60},
    {"input": "Works as advertised, good value", "label": "POSITIVE", "score": 0.78},
]

def build_few_shot_prompt(query: str, examples: list, n: int = 3) -> str:
    """Build few-shot prompt từ example pool."""
    selected = examples[:n]  # Simplified: production dùng embedding similarity
    
    prompt_parts = []
    for i, ex in enumerate(selected, 1):
        prompt_parts.append(
            f"# Example {i}:\n"
            f"Review: \"{ex['input']}\"\n"
            f"Label: {ex['label']}\n"
            f"Score: {ex['score']}"
        )
    
    prompt_parts.append(f"\n# Analyze:\nReview: \"{query}\"\nLabel:\nScore:")
    return "\n\n".join(prompt_parts)

test_query = "Product quality exceeded expectations but packaging was terrible"
prompt = build_few_shot_prompt(test_query, EXAMPLE_POOL)
print("Generated prompt:")
print(prompt)
print("\n" + "=" * 40)
result = ask(prompt, system_msg="You are a sentiment analysis expert.")
print(f"Result: {result}")

---
## 3. 🔗 Chain-of-Thought (CoT) Prompting

CoT yêu cầu model "suy nghĩ to" (think out loud) trước khi đưa ra câu trả lời. Leverages in-context learning và buộc model decompose complex problems.

**Research:** CoT cải thiện accuracy trên arithmetic reasoning từ ~18% → ~57% (Wei et al., 2022).

In [ ]:
# 3.1 Zero-shot CoT: "Let's think step by step"

# ❌ Không có CoT
no_cot = """
An AI agent needs to process 1,000 requests per day.
Each request consumes an average of 500 tokens (input + output).
GPT-4 Turbo is priced at $0.01/1K input tokens and $0.03/1K output tokens.
Assume an input:output ratio of 60:40.
What is the total monthly cost in USD?
"""

# ✅ Với Zero-shot CoT
with_cot = no_cot + "\nLet's think step by step before answering."

print("=" * 50)
print("WITHOUT CoT:")
print("=" * 50)
print(ask(no_cot))

print("\n" + "=" * 50)
print("WITH Zero-shot CoT:")
print("=" * 50)
print(ask(with_cot))

In [ ]:
# 3.2 Few-shot CoT — Dạy reasoning pattern cụ thể

cot_prompt = """
Q: If an agent calls 3 tools in parallel, each taking 2 seconds, but one tool
times out after 5 seconds, what is the total elapsed time?

A: Let's reason step by step:
1. Three tools run in parallel so total time = max(time of each tool)
2. Tool 1: 2 seconds, Tool 2: 2 seconds, Tool 3: timeout = 5 seconds
3. max(2, 2, 5) = 5 seconds
4. Adding retry logic: typically +2 seconds overhead
Result: ~7 seconds with retry, or 5 seconds if fail-fast

---

Q: A vector database stores 1M embeddings of dimension 1536 (float32).
If we add HNSW index (typically 1.5x memory overhead), how much RAM is needed?

A: Let's reason step by step:
"""

result = ask(cot_prompt)
print(result)

In [ ]:
# 3.3 Self-Consistency (SC) — Advanced CoT
# Chạy cùng prompt N lần, lấy majority vote
# Ideal cho tasks có 1 đáp án đúng rõ ràng

import asyncio

async def self_consistency(prompt: str, n_samples: int = 5, temperature: float = 0.7) -> dict:
    """Run prompt n times, return majority answer + distribution."""
    async_client = AsyncAnthropic()
    
    async def get_answer():
        response = await async_client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=500,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        text = response.content[0].text.strip()
        # Extract final answer (last line)
        return text.split("\n")[-1].strip()
    
    tasks = [get_answer() for _ in range(n_samples)]
    answers = await asyncio.gather(*tasks)
    
    distribution = Counter(answers)
    majority = distribution.most_common(1)[0][0]
    confidence = distribution[majority] / n_samples
    
    return {
        "majority_answer": majority,
        "confidence": confidence,
        "all_answers": list(answers),
        "distribution": dict(distribution)
    }

# Test
sc_prompt = """
A system has:
- 3 microservices, each with 99.9% uptime (monthly)
- All 3 must be running for the system to work

What is the system's overall uptime percentage?
Let's think step by step, then give ONLY the final percentage on the last line.
"""

# Chạy async trong Jupyter
result = await self_consistency(sc_prompt, n_samples=5)
print(f"✅ Majority Answer: {result['majority_answer']}")
print(f"📊 Confidence: {result['confidence']:.0%}")
print(f"📈 Distribution: {result['distribution']}")
print(f"\n🔍 All answers: {result['all_answers']}")

In [ ]:
# 3.4 Tree-of-Thought (ToT)
# Khám phá nhiều reasoning branches, evaluate chúng

tot_prompt = """
Problem: Design a memory architecture for an AI Agent that must handle
10,000 sessions per day, each session containing 50 turns.

Propose 3 DISTINCT approaches. For each:
- Give a 2-sentence description
- List 2 Pros and 2 Cons
- Score it /10 for this use case

Format:
## Approach 1: [Name]
Description: ...
Pros: ...
Cons: ...
Score: .../10

## Approach 2: [Name]
...

## Approach 3: [Name]
...

## Final Recommendation
[Which approach and why, in 2 sentences]
"""

result = ask(tot_prompt, max_tokens=1500)
print(result)

### 🎯 Khi nào dùng kỹ thuật CoT nào?

| Kỹ thuật | Khi dùng | Trade-off |
|----------|----------|----------|
| **Zero-shot CoT** | Bài toán reasoning đơn giản | +latency nhỏ |
| **Few-shot CoT** | Reasoning domain-specific | Cần curate examples |
| **Self-Consistency** | High-accuracy requirements | 5x latency & cost |
| **Tree-of-Thought** | Tìm giải pháp tốt nhất trong không gian lớn | Phức tạp nhất |

---
## 4. 🎭 Role Prompting & System Prompt Design

In [ ]:
# 4.1 Basic Role Prompting: Vague vs Specific

code_to_review = """
async def get_user(user_id: int, db: Session = Depends(get_db)):
    user = db.query(User).filter(User.id == user_id).first()
    return user
"""

# ❌ BAD: Vague role
bad_system = "You are a helpful AI assistant."

# ✅ GOOD: Specific role with behavior rules
good_system = """
You are a Senior Python Engineer with 10 years of experience building
distributed systems and AI infrastructure at FAANG-scale companies.

When reviewing code, you ALWAYS:
1. Identify specific performance bottlenecks with line references
2. Propose improvements with concrete code examples
3. Explain trade-offs clearly (e.g., speed vs. memory vs. maintainability)
4. Prioritize: security > correctness > performance > style

You do NOT:
- Praise code unnecessarily before critiquing
- Give vague suggestions like "consider refactoring"
- Explain basic Python concepts to the reviewer
"""

question = f"Review this FastAPI endpoint:\n```python{code_to_review}```"

print("=" * 50)
print("BAD (vague role):")
print("=" * 50)
print(ask(question, system_msg=bad_system, max_tokens=300))

print("\n" + "=" * 50)
print("GOOD (specific role):")
print("=" * 50)
print(ask(question, system_msg=good_system, max_tokens=500))

In [ ]:
# 4.2 Anatomy: Professional System Prompt cho AI Agent

AGENT_SYSTEM_PROMPT = """
# IDENTITY
You are CodeReview Agent, an expert Python code reviewer.
You have 15 years of experience and have reviewed over 50,000 pull requests.

# OBJECTIVE
Task: Analyze the provided code and deliver actionable feedback.
Audience: Experienced Python developers — do NOT explain basics.

# CAPABILITIES
You may reference these conceptual tools in your review:
- Performance analysis (Big-O, DB query plans)
- Security scanning (OWASP Top 10)
- Type safety and Pydantic validation

# CONSTRAINTS
- Do NOT auto-fix code unless explicitly asked
- Do NOT review architecture outside the provided code scope
- Limit feedback to a maximum of 10 items per review
- If tests are missing, ALWAYS list under Critical Issues

# OUTPUT FORMAT
Every review MUST follow this structure exactly:

## Summary
[1-2 sentence overview]

## Critical Issues (must fix)
- [issue]: [explanation] → [concrete suggestion]

## Suggestions (should fix)
- [suggestion]: [reason]

## Positives
- [what was done well]

# BEHAVIOR
- If no critical issues: explicitly state 'Approved with minor suggestions'
- Priority order: security > correctness > performance > style
"""

test_code = """
import sqlite3

def get_user_by_name(name: str):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query = f"SELECT * FROM users WHERE name = '{name}'"
    cursor.execute(query)
    return cursor.fetchall()
"""

result = ask(f"Review this code:\n```python{test_code}```",
             system_msg=AGENT_SYSTEM_PROMPT,
             max_tokens=800)
print(result)

In [ ]:
# 4.3 Multi-persona cho Multi-agent Systems
# Mỗi agent cần persona riêng biệt để tránh role confusion

ORCHESTRATOR_PROMPT = """
You are the Orchestrator Agent.

Your ONLY responsibilities are:
1. Receive tasks from the user
2. Decompose them into clearly defined subtasks
3. Assign subtasks to specialist agents (Research or Implementation)
4. Synthesize results into a coherent final response

You do NOT execute subtasks yourself. You ONLY coordinate.

Output format when decomposing:
{"subtasks": [{"agent": "research|implementation", "task": "...", "priority": 1}]}
"""

RESEARCHER_PROMPT = """
You are the Research Agent.

Your ONLY responsibilities:
- Search for information when requested by the Orchestrator
- Return raw information — do NOT interpret, recommend, or implement

Output format (STRICT JSON):
{"source": "...", "content": "...", "confidence_score": 0.0-1.0}
"""

IMPLEMENTER_PROMPT = """
You are the Implementation Agent.

Your ONLY responsibilities:
- Write and review code when requested by the Orchestrator
- Return working, production-ready code with brief explanation
- Do NOT make design decisions beyond the task scope
"""

# Demo: Orchestrator decomposing a task
user_task = "Build a Python function that validates email addresses and checks if they're from a corporate domain (not Gmail, Yahoo, etc.)"

decomposition = ask(f"Decompose this task: {user_task}", system_msg=ORCHESTRATOR_PROMPT)
print("Orchestrator decomposition:")
print(decomposition)

---
## 5. 📐 Structured Prompting & Templates

In [ ]:
# 5.1 Jinja2 Template Engine cho Prompts
# Trong production, prompts KHÔNG BAO GIỜ hardcode

from jinja2 import Template
from dataclasses import dataclass
from typing import Optional

@dataclass
class PromptConfig:
    role: str
    task: str
    context: str
    output_format: str
    examples: Optional[list] = None
    constraints: Optional[list] = None

AGENT_TEMPLATE = Template("""
# ROLE
{{ role }}

# TASK
{{ task }}

# CONTEXT
{{ context }}

{% if constraints %}
# CONSTRAINTS
{% for c in constraints %}- {{ c }}
{% endfor %}{% endif %}

{% if examples %}
# EXAMPLES
{% for ex in examples %}
Input: {{ ex.input }}
Output: {{ ex.output }}
{% endfor %}{% endif %}

# OUTPUT FORMAT
{{ output_format }}
""")

# Usage
config = PromptConfig(
    role="Senior Code Reviewer specializing in Python security",
    task="Review the provided Python code for security vulnerabilities",
    context="Codebase: FastAPI service, Python 3.12, handles financial transactions",
    output_format='JSON with fields: {"vulnerabilities": [...], "severity": "low|medium|high|critical", "fix": "..."}',
    constraints=[
        "Do not suggest a full rewrite",
        "Max 5 critical issues",
        "Include CVE references where applicable"
    ]
)

rendered_prompt = AGENT_TEMPLATE.render(
    role=config.role,
    task=config.task,
    context=config.context,
    output_format=config.output_format,
    constraints=config.constraints,
    examples=config.examples
)

print("Rendered prompt:")
print(rendered_prompt)

In [ ]:
# 5.2 XML Tags — Claude đặc biệt hiểu tốt XML tags

xml_structured_prompt = """
<task>
Analyze the following Python code snippet and identify bugs.
</task>

<context>
This is a FastAPI endpoint handler in a production system.
The service processes 10,000 requests per minute.
</context>

<code>
async def get_user(user_id: int, db: Session = Depends(get_db)):
    user = db.query(User).filter(User.id == user_id).first()
    return user
</code>

<output_format>
Respond ONLY with valid JSON:
{"bugs": ["..."], "severity": "low|medium|high", "fix": "..."}
</output_format>
"""

result = ask(xml_structured_prompt)
print("Raw output:")
print(result)

# Parse JSON
try:
    parsed = json.loads(result)
    print("\n✅ Valid JSON parsed:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print("\n⚠️ Output is not pure JSON — cần thêm constraints")

In [ ]:
# 5.3 Prompt Chaining Pattern
# Bẻ task phức tạp thành nhiều prompts nhỏ hơn

def analyze_codebase_with_chaining(code: str) -> dict:
    """3-step prompt chain: Extract → Analyze → Recommend"""
    
    print("⛓️  Step 1: Extracting structure...")
    step1 = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        system="You are a code parser. Extract code structure as concise JSON. Include: functions, classes, imports, potential issues.",
        messages=[{"role": "user", "content": f"Parse this code:\n```python\n{code}\n```"}]
    )
    structure = step1.content[0].text
    print(f"   Structure extracted: {len(structure)} chars")
    
    print("⛓️  Step 2: Analyzing for security issues...")
    step2 = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        system="You are a security expert. Given code structure JSON, identify specific vulnerabilities. Be concise and technical.",
        messages=[{"role": "user", "content": f"Code structure:\n{structure}\n\nIdentify vulnerabilities:"}]
    )
    analysis = step2.content[0].text
    print(f"   Analysis complete: {len(analysis)} chars")
    
    print("⛓️  Step 3: Generating action plan...")
    step3 = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system="You are a senior architect. Given a vulnerability analysis, produce a prioritized action plan with concrete code fixes.",
        messages=[{"role": "user", "content": f"Vulnerabilities found:\n{analysis}\n\nGenerate prioritized action plan with code examples:"}]
    )
    plan = step3.content[0].text
    
    print("✅ Chain complete!")
    return {"structure": structure, "analysis": analysis, "action_plan": plan}

vulnerable_code = """
import sqlite3
import os

def authenticate(username: str, password: str) -> dict:
    conn = sqlite3.connect('app.db')
    cursor = conn.cursor()
    query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
    cursor.execute(query)
    user = cursor.fetchone()
    if user:
        token = os.environ.get('SECRET_KEY', 'default_secret') + username
        return {"token": token, "user": user}
    return {}
"""

results = analyze_codebase_with_chaining(vulnerable_code)
print("\n" + "=" * 50)
print("ACTION PLAN:")
print("=" * 50)
print(results["action_plan"])

---
## 6. 🤖 Advanced Techniques for AI Agents

In [ ]:
# 6.1 ReAct Prompting (Reason + Act)
# Backbone của nhiều AI Agent frameworks
# Kết hợp reasoning (CoT) với action (tool use)

# Define mock tools
def search(query: str) -> str:
    """Mock search tool."""
    mock_results = {
        "Claude claude-sonnet-4-6 pricing": "Claude claude-sonnet-4-6 pricing: $3/1M input, $15/1M output tokens (as of 2024)",
        "GPT-4o pricing": "GPT-4o pricing: $5/1M input, $15/1M output tokens",
    }
    for key, val in mock_results.items():
        if any(word.lower() in query.lower() for word in key.split()):
            return val
    return f"Search results for '{query}': [No specific results found]"

def calculate(expr: str) -> str:
    """Safe math evaluator."""
    try:
        result = eval(expr, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

TOOLS = {"search": search, "calculate": calculate}

REACT_SYSTEM = """
Solve tasks using the pattern: Thought → Action → Observation

EXACT format required:
Thought: [Reasoning about what to do next]
Action: tool_name("parameter")
Observation: [Result returned by the tool]
... (repeat as needed)
Thought: I now have enough information to answer
Final Answer: [Final response to the user]

Available tools:
- search("query") → Finds information about pricing, facts, documentation
- calculate("math_expr") → Evaluates a Python math expression

IMPORTANT: After each Action, I will provide the Observation. Wait for it.
Stop when you can write a Final Answer.
"""

def run_react_agent(user_query: str, max_iterations: int = 6) -> str:
    """Simple ReAct agent loop."""
    import re
    
    messages = [{"role": "user", "content": user_query}]
    
    for iteration in range(max_iterations):
        print(f"\n🔄 Iteration {iteration + 1}")
        
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=500,
            system=REACT_SYSTEM,
            messages=messages,
            stop_sequences=["Observation:"]
        )
        
        agent_output = response.content[0].text
        print(agent_output)
        
        # Check for Final Answer
        if "Final Answer:" in agent_output:
            final = agent_output.split("Final Answer:")[1].strip()
            return final
        
        # Parse and execute Action
        action_match = re.search(r'Action: (\w+)\("?([^"\)]+)"?\)', agent_output)
        if action_match:
            tool_name = action_match.group(1)
            tool_param = action_match.group(2)
            
            if tool_name in TOOLS:
                observation = TOOLS[tool_name](tool_param)
            else:
                observation = f"Error: Tool '{tool_name}' not found"
            
            print(f"Observation: {observation}")
            
            # Add to conversation
            messages.append({"role": "assistant", "content": agent_output})
            messages.append({"role": "user", "content": f"Observation: {observation}"})
        else:
            break
    
    return "Max iterations reached"

print("=" * 60)
print("ReAct Agent Demo")
print("=" * 60)
query = "What would it cost to run 500,000 Claude claude-sonnet-4-6 requests per month at 1000 tokens each (50% input, 50% output)?"
answer = run_react_agent(query)
print(f"\n✅ FINAL ANSWER: {answer}")

In [ ]:
# 6.2 Meta-Prompting: Dùng LLM để generate prompts
# Hữu ích khi không chắc prompt nào hiệu quả nhất

def generate_optimized_prompt(
    task_description: str,
    output_spec: str,
    constraints: str
) -> str:
    """Use LLM to generate an optimized prompt for a given task."""
    
    meta_prompt = f"""
    You are an expert Prompt Engineer specializing in Claude claude-sonnet-4-6.
    
    Create an optimal system prompt for this use case:
    
    Task: {task_description}
    Required output: {output_spec}
    Constraints: {constraints}
    
    Generate a complete system prompt that includes:
    1. Identity & role definition
    2. Detailed behavioral instructions
    3. 2 concrete few-shot examples
    4. Exact output format specification using XML tags
    5. Edge case handling rules
    
    Then briefly explain (2-3 sentences) your key design decisions.
    """
    
    return ask(meta_prompt, max_tokens=1500)

# Generate a prompt for a customer support agent
generated = generate_optimized_prompt(
    task_description="Handle customer complaints about delayed e-commerce orders",
    output_spec="Empathetic response + action plan + timeline estimate",
    constraints="Must not promise refunds without manager approval; escalate if customer is hostile"
)

print(generated)

In [ ]:
# 6.3 Self-Critique và Reflection
# Dạy model tự đánh giá và cải thiện output của chính nó

def self_critique_loop(task: str, criteria: list[str]) -> dict:
    """Generate → Critique → Improve loop."""
    
    # Step 1: Generate initial response
    print("📝 Step 1: Initial generation...")
    initial = ask(task, max_tokens=600)
    print(f"Initial ({len(initial)} chars):\n{initial[:200]}...")
    
    # Step 2: Self-critique
    print("\n🔍 Step 2: Self-critique...")
    criteria_str = "\n".join(f"- {c}" for c in criteria)
    critique_prompt = f"""
    You wrote this response:
    ---
    {initial}
    ---
    
    Critically evaluate it against these criteria:
    {criteria_str}
    
    For each criterion, rate it 1-5 and explain WHY (be harsh and specific).
    Then list the 3 most important improvements needed.
    """
    critique = ask(critique_prompt, max_tokens=600)
    print(f"Critique:\n{critique[:300]}...")
    
    # Step 3: Improved response
    print("\n✨ Step 3: Improved response...")
    improve_prompt = f"""
    Original task: {task}
    
    Your initial response had these issues:
    {critique}
    
    Write a significantly improved response that addresses ALL identified issues.
    """
    improved = ask(improve_prompt, max_tokens=800)
    
    return {"initial": initial, "critique": critique, "improved": improved}

# Test
task = "Explain why async/await is important for AI agent development in Python. Target audience: experienced developers."
criteria = [
    "Accuracy: Are all technical claims correct?",
    "Specificity: Does it avoid vague generalizations?",
    "Actionability: Can a developer apply this immediately?",
    "Conciseness: Is every sentence necessary?"
]

results = self_critique_loop(task, criteria)
print("\n" + "=" * 50)
print("FINAL IMPROVED RESPONSE:")
print("=" * 50)
print(results["improved"])

---
## 7. 📦 Output Control & JSON Mode

In [ ]:
# 7.1 Đảm bảo parseable JSON output
# Trong AI Agents, structured output là bắt buộc

from pydantic import BaseModel, Field
from typing import List, Literal

class CodeIssue(BaseModel):
    line: Optional[int]
    issue_type: Literal["security", "performance", "correctness", "style"]
    description: str
    suggestion: str

class CodeReviewResult(BaseModel):
    overall_severity: Literal["low", "medium", "high", "critical"]
    issues: List[CodeIssue]
    approved: bool
    summary: str

def get_structured_code_review(code: str) -> CodeReviewResult:
    """Returns type-safe, validated code review."""
    
    schema_str = json.dumps(CodeReviewResult.model_json_schema(), indent=2)
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        system=f"""You are a code reviewer.
        Return ONLY valid JSON conforming to this schema:
        {schema_str}
        
        Rules:
        - Output JSON only — no additional text or markdown fences
        - If data unavailable, use null; never omit required fields
        - Set approved=true ONLY if no security or correctness issues""",
        messages=[{"role": "user", "content": f"Review this code:\n```python\n{code}\n```"}]
    )
    
    raw = response.content[0].text.strip()
    
    # Clean markdown fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): 
            raw = raw[4:]
    
    parsed = json.loads(raw.strip())
    return CodeReviewResult(**parsed)

# Test with vulnerable code
bad_code = """
def process_payment(user_input: str, amount: float):
    # Execute user-provided SQL
    db.execute(f"INSERT INTO payments VALUES ({user_input}, {amount})")
    return True
"""

review = get_structured_code_review(bad_code)
print(f"✅ Parsed CodeReviewResult:")
print(f"  Severity: {review.overall_severity}")
print(f"  Approved: {review.approved}")
print(f"  Summary: {review.summary}")
print(f"  Issues ({len(review.issues)}):")
for issue in review.issues:
    print(f"    [{issue.issue_type.upper()}] {issue.description}")
    print(f"    → {issue.suggestion}")

In [ ]:
# 7.2 Robust JSON Extraction — Xử lý khi model không tuân thủ format

import re

def robust_json_extract(text: str) -> dict:
    """Extract JSON từ model output, dù có markdown fences hay không."""
    
    # Strategy 1: Try direct parse
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    
    # Strategy 2: Extract from markdown code block
    match = re.search(r'```(?:json)?\s*([\s\S]+?)```', text)
    if match:
        try:
            return json.loads(match.group(1).strip())
        except json.JSONDecodeError:
            pass
    
    # Strategy 3: Find JSON object/array in text
    match = re.search(r'(\{[\s\S]+\}|\[[\s\S]+\])', text)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    
    raise ValueError(f"No valid JSON found in output: {text[:200]}")

# Test với các cases khác nhau
test_cases = [
    '{"key": "value"}',  # Direct JSON
    '```json\n{"key": "value"}\n```',  # Markdown fenced
    'Here is the result:\n{"key": "value"}\n\nHope that helps!',  # Embedded
]

for tc in test_cases:
    try:
        result = robust_json_extract(tc)
        print(f"✅ Parsed: {result}")
    except ValueError as e:
        print(f"❌ Failed: {e}")

---
## 8. 📊 Prompt Evaluation & A/B Testing

In [ ]:
# 8.1 Prompt Evaluation Framework
# Professional PE cần eval framework — không phải intuition

@dataclass
class EvalCase:
    input_text: str
    expected_output: str
    weight: float = 1.0
    description: str = ""

class PromptEvaluator:
    def __init__(self, eval_cases: list[EvalCase]):
        self.cases = eval_cases
    
    def _run_prompt(self, system_prompt: str, user_input: str) -> str:
        return ask(user_input, system_msg=system_prompt, max_tokens=300)
    
    def _llm_score(self, actual: str, expected: str) -> float:
        """Use LLM as judge to score output."""
        score_prompt = f"""
        Rate how well the ACTUAL output matches the EXPECTED output on a scale of 0.0 to 1.0.
        
        Expected: {expected}
        Actual: {actual}
        
        Consider: semantic similarity, format compliance, key information present.
        Respond with ONLY a number between 0.0 and 1.0. Nothing else.
        """
        score_text = ask(score_prompt, max_tokens=10)
        try:
            return float(score_text.strip())
        except:
            return 0.5
    
    def evaluate(self, system_prompt: str, n_runs: int = 2) -> dict:
        """Evaluate a prompt across all test cases."""
        all_scores = []
        case_results = []
        
        for case in self.cases:
            run_scores = []
            for _ in range(n_runs):
                actual = self._run_prompt(system_prompt, case.input_text)
                score = self._llm_score(actual, case.expected_output)
                run_scores.append(score * case.weight)
            
            avg_score = statistics.mean(run_scores)
            all_scores.append(avg_score)
            case_results.append({
                "case": case.description or case.input_text[:50],
                "score": avg_score,
                "pass": avg_score >= 0.7
            })
        
        return {
            "mean_score": statistics.mean(all_scores),
            "std_dev": statistics.stdev(all_scores) if len(all_scores) > 1 else 0,
            "min_score": min(all_scores),
            "pass_rate": sum(1 for s in all_scores if s >= 0.7) / len(all_scores),
            "case_results": case_results
        }

# Define eval cases
sentiment_cases = [
    EvalCase("Excellent product, will buy again!", "POSITIVE", description="Clear positive"),
    EvalCase("Terrible, broke after 2 days", "NEGATIVE", description="Clear negative"),
    EvalCase("It's okay I guess", "NEUTRAL", description="Ambiguous neutral"),
    EvalCase("Great quality but overpriced", "MIXED", weight=1.5, description="Mixed sentiment"),
]

evaluator = PromptEvaluator(sentiment_cases)

# Compare two prompts
prompt_v1 = "Classify sentiment as POSITIVE, NEGATIVE, NEUTRAL, or MIXED. Output just the label."
prompt_v2 = """You are a sentiment analysis expert. Classify the review sentiment.
Output EXACTLY one of: POSITIVE, NEGATIVE, NEUTRAL, MIXED
No explanation needed, just the label."""

print("Evaluating Prompt V1...")
result_v1 = evaluator.evaluate(prompt_v1, n_runs=1)
print(f"V1 Score: {result_v1['mean_score']:.2f} | Pass Rate: {result_v1['pass_rate']:.0%}")

print("\nEvaluating Prompt V2...")
result_v2 = evaluator.evaluate(prompt_v2, n_runs=1)
print(f"V2 Score: {result_v2['mean_score']:.2f} | Pass Rate: {result_v2['pass_rate']:.0%}")

winner = "V1" if result_v1['mean_score'] > result_v2['mean_score'] else "V2"
print(f"\n🏆 Winner: Prompt {winner}")

In [ ]:
# 8.2 Common Anti-patterns — Học từ sai lầm

print("🚫 PROMPT ANTI-PATTERNS TO AVOID\n")
print("=" * 60)

antipatterns = [
    {
        "name": "1. Vague Instructions",
        "bad": "Be good and helpful",
        "good": "When asked about code, always provide: (1) explanation, (2) fixed code, (3) test case",
        "why": "Specific instructions = predictable behavior"
    },
    {
        "name": "2. Negative-Only Constraints",
        "bad": "Don't give wrong answers. Don't be verbose.",
        "good": "Always verify before answering. Keep responses under 200 words.",
        "why": "Tell the model WHAT TO DO, not what to avoid"
    },
    {
        "name": "3. Overloaded Single Prompt",
        "bad": "Translate, summarize, improve grammar, add examples, and format as HTML",
        "good": "Use prompt chaining: translate → summarize → improve → format",
        "why": "Models handle early tasks better than later ones in long prompts"
    },
    {
        "name": "4. Inconsistent Few-shot Examples",
        "bad": "Example 1 uses Label:, Example 2 uses Category:, Example 3 uses Type:",
        "good": "All examples use identical field names and separators",
        "why": "Inconsistency confuses the model → unstable output format"
    },
    {
        "name": "5. No Output Constraints",
        "bad": "Tell me about Python async programming",
        "good": "In exactly 3 bullet points (max 20 words each), explain Python async key concepts",
        "why": "Unconstrained output = unpredictable length, format, and token cost"
    }
]

for ap in antipatterns:
    print(f"\n{ap['name']}")
    print(f"  ❌ Bad:  {ap['bad']}")
    print(f"  ✅ Good: {ap['good']}")
    print(f"  💡 Why:  {ap['why']}")

---
## 9. 🔧 Model-specific Best Practices

In [ ]:
# 9.1 Claude (Anthropic) — Best Practices

# ✅ XML Tags — Claude được fine-tune để hiểu XML tags cực tốt
claude_optimized = """
<role>
Senior Python engineer with 10+ years experience
</role>

<task>
Review the code below and identify the top 3 issues
</task>

<code>
def get_data(url):
    import requests
    return requests.get(url).json()
</code>

<format>
Return JSON: {"issues": [{"priority": 1-3, "issue": "...", "fix": "..."}]}
</format>
"""

result = ask(claude_optimized)
print("Claude with XML tags:")
print(result)

# ✅ Anti-sycophancy technique
print("\n" + "=" * 40)
print("Anti-sycophancy:")
anti_syco = ask(
    "My plan: use SQLite for a service handling 100K concurrent users. What do you think?",
    system_msg="You are a database expert. Critically challenge architectural decisions. Do NOT be politely agreeable if there are real problems."
)
print(anti_syco)

In [ ]:
# 9.2 Extended Thinking Mode (Claude)
# Dùng cho complex multi-step reasoning

# Note: Extended thinking cần claude-sonnet-4-6 và budget_tokens
def ask_with_thinking(prompt: str, budget_tokens: int = 5000) -> dict:
    """Use Claude's extended thinking for complex reasoning."""
    try:
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=budget_tokens + 2000,
            temperature=1,  # Required for extended thinking
            thinking={
                "type": "enabled",
                "budget_tokens": budget_tokens
            },
            messages=[{"role": "user", "content": prompt}]
        )
        
        thinking_content = ""
        text_content = ""
        
        for block in response.content:
            if block.type == "thinking":
                thinking_content = block.thinking
            elif block.type == "text":
                text_content = block.text
        
        return {"thinking": thinking_content, "answer": text_content}
    except Exception as e:
        return {"error": str(e), "thinking": "", "answer": ""}

complex_problem = """
Design a rate limiting strategy for an AI Agent API that:
1. Handles 10,000 requests/second at peak
2. Has 3 tiers: Free (100/day), Pro (10K/day), Enterprise (unlimited)
3. Must prevent single users from monopolizing resources
4. Should have minimal latency impact (<1ms overhead)

Recommend specific algorithms and data structures with trade-off analysis.
"""

result = ask_with_thinking(complex_problem, budget_tokens=3000)

if "error" in result:
    print(f"Extended thinking not available: {result['error']}")
    print("\nFallback to regular response:")
    print(ask(complex_problem, max_tokens=800))
else:
    print("🧠 Thinking process (excerpt):")
    print(result['thinking'][:500] + "...")
    print("\n" + "=" * 50)
    print("💬 Final Answer:")
    print(result['answer'])

### Model Comparison Cheatsheet

| Model | Tip tốt nhất | Lưu ý quan trọng |
|-------|--------------|------------------|
| **Claude** | XML tags cho structure; anti-sycophancy prompt | Tránh pre-fill Assistant trừ khi cần control prefix |
| **GPT-4o** | All instructions trong system message | Dùng `response_format={"type": "json_object"}` cho JSON |
| **Gemini 1.5** | Pass images/video trực tiếp; 1M context | Configure `safety_settings` explicitly |
| **Llama 3** | Embed instructions trong user message | Temperature mặc định cao — giảm xuống 0.1–0.3 cho production |

---
## 10. 🏗️ Capstone: Production Prompt Library

In [ ]:
# SmartPromptManager — Production-ready prompt management

from pydantic import BaseModel
import hashlib

class PromptVersion(BaseModel):
    name: str
    version: str
    system_template: str
    user_template: str
    variables: list[str]
    model: str = "claude-sonnet-4-6"
    max_tokens: int = 1000
    temperature: float = 0.1
    tags: list[str] = []

class SmartPromptManager:
    def __init__(self):
        self.client = Anthropic()
        self.prompts: dict[str, PromptVersion] = {}
        self._cache: dict[str, str] = {}
        self._usage: dict[str, int] = {}
    
    def register(self, prompt: PromptVersion):
        """Register a prompt version."""
        self.prompts[prompt.name] = prompt
        self._usage[prompt.name] = 0
        print(f"✅ Registered: {prompt.name} v{prompt.version}")
    
    def _render(self, template: str, variables: dict) -> str:
        """Simple template rendering."""
        result = template
        for key, val in variables.items():
            result = result.replace(f"{{{{{key}}}}}", str(val))
        return result
    
    def _cache_key(self, name: str, variables: dict) -> str:
        payload = f"{name}:{json.dumps(variables, sort_keys=True)}"
        return hashlib.md5(payload.encode()).hexdigest()
    
    def run(
        self, 
        prompt_name: str, 
        use_cache: bool = True, 
        **kwargs
    ) -> dict:
        """Execute a registered prompt."""
        if prompt_name not in self.prompts:
            raise ValueError(f"Prompt '{prompt_name}' not registered")
        
        pv = self.prompts[prompt_name]
        
        # Check required variables
        missing = [v for v in pv.variables if v not in kwargs]
        if missing:
            raise ValueError(f"Missing variables: {missing}")
        
        # Check cache
        cache_key = self._cache_key(prompt_name, kwargs)
        if use_cache and cache_key in self._cache:
            print(f"  💾 Cache hit for {prompt_name}")
            return {"output": self._cache[cache_key], "cached": True}
        
        # Render templates
        system = self._render(pv.system_template, kwargs)
        user = self._render(pv.user_template, kwargs)
        
        # Call API
        response = self.client.messages.create(
            model=pv.model,
            max_tokens=pv.max_tokens,
            temperature=pv.temperature,
            system=system,
            messages=[{"role": "user", "content": user}]
        )
        
        output = response.content[0].text
        
        # Cache result
        self._cache[cache_key] = output
        self._usage[prompt_name] += 1
        
        return {
            "output": output,
            "cached": False,
            "tokens_used": response.usage.input_tokens + response.usage.output_tokens
        }
    
    def stats(self) -> dict:
        return {
            "registered_prompts": len(self.prompts),
            "cache_size": len(self._cache),
            "usage": self._usage
        }

# Initialize and register prompts
pm = SmartPromptManager()

pm.register(PromptVersion(
    name="code_review",
    version="1.2.0",
    system_template="""You are a Senior {{language}} Engineer.
Review code for: security, performance, correctness.
Output JSON: {"issues": [{"type": "...", "description": "...", "fix": "..."}], "approved": true/false}""",
    user_template="Review this {{language}} code:\n```\n{{code}}\n```",
    variables=["language", "code"],
    temperature=0.0
))

pm.register(PromptVersion(
    name="summarize",
    version="1.0.0",
    system_template="You are a technical writer. Summarize content for {{audience}} in {{style}} style.",
    user_template="Summarize in {{max_words}} words:\n{{content}}",
    variables=["audience", "style", "max_words", "content"],
))

# Use the manager
print("\n📋 Running code_review prompt...")
result = pm.run(
    "code_review",
    language="Python",
    code="def add(a, b): return a + b"
)
print(f"Output: {result['output'][:200]}...")
print(f"Tokens used: {result.get('tokens_used', 'N/A')}")

# Same call again — should hit cache
print("\n📋 Running same prompt again (cache test)...")
result2 = pm.run(
    "code_review",
    language="Python",
    code="def add(a, b): return a + b"
)
print(f"Cached: {result2['cached']}")

print(f"\n📊 Stats: {pm.stats()}")

---
## 11. 🎓 Summary & Cheatsheet

In [ ]:
# Final Summary — In ra để reference

summary = """
╔══════════════════════════════════════════════════════════════╗
║         PROMPT ENGINEERING CHEATSHEET — AI Agents           ║
╚══════════════════════════════════════════════════════════════╝

┌─ TECHNIQUE SELECTION ──────────────────────────────────────┐
│ Simple task, model knows well    → Zero-shot               │
│ Need specific format/pattern     → Few-shot (3-5 examples) │
│ Complex reasoning, single answer → CoT + Self-Consistency  │
│ Find best among many options     → Tree-of-Thought         │
│ Agent with tools                 → ReAct pattern           │
│ Unknown best prompt              → Meta-Prompting          │
│ High-quality output critical     → Self-Critique loop      │
└────────────────────────────────────────────────────────────┘

┌─ SYSTEM PROMPT ANATOMY ────────────────────────────────────┐
│ IDENTITY  → Who the agent is (specific, not vague)        │
│ OBJECTIVE → What exactly it does                          │
│ CAPABILITIES → Available tools/actions                    │
│ CONSTRAINTS → What NOT to do (state as positive rules)    │
│ OUTPUT FORMAT → Exact structure of response               │
│ BEHAVIOR → Edge case handling rules                       │
└────────────────────────────────────────────────────────────┘

┌─ PRODUCTION RULES ─────────────────────────────────────────┐
│ 1. NEVER hardcode prompts — use Jinja2 templates           │
│ 2. ALWAYS eval prompts with test cases, not gut feeling    │
│ 3. For JSON output: use XML tags + Pydantic validation     │
│ 4. Use prompt chaining for complex multi-step tasks        │
│ 5. Cache identical prompt+input pairs to save cost         │
│ 6. Version control your prompts like you do code           │
│ 7. A/B test: accuracy, latency, cost, consistency          │
└────────────────────────────────────────────────────────────┘

┌─ CLAUDE-SPECIFIC ──────────────────────────────────────────┐
│ • Use XML tags for every structured section                │
│ • End with Human turn; only pre-fill when controlling prefix│
│ • Add "Critically challenge" to prevent sycophancy         │
│ • Extended thinking for complex multi-step reasoning       │
└────────────────────────────────────────────────────────────┘

┌─ TOP ANTI-PATTERNS TO AVOID ───────────────────────────────┐
│ ❌ Vague roles ("helpful assistant")                       │
│ ❌ Negative-only constraints ("don't do X")                │
│ ❌ Overloaded single prompts (5+ tasks in 1 prompt)        │
│ ❌ Inconsistent few-shot examples                          │
│ ❌ No output constraints (format, length)                  │
└────────────────────────────────────────────────────────────┘
"""
print(summary)

---
## 12. 🏋️ Exercises

Hoàn thành các bài tập sau trước khi chuyển sang Phase tiếp theo:

### Exercise 1 — Few-shot Improvement
Tạo few-shot prompt để classify intent của customer support messages thành: `REFUND`, `SHIPPING`, `TECHNICAL`, `BILLING`, `OTHER`. Test với 10 messages khác nhau.

### Exercise 2 — ReAct Agent
Mở rộng ReAct agent ở Section 6.1 để thêm tool `read_file(path)` và `write_file(path, content)`. Test với task: "Read config.json, update the 'max_retries' field to 5, and save back."

### Exercise 3 — Prompt Evaluator
Tạo 10 eval cases cho một prompt của bạn. So sánh 3 versions của prompt đó. Implement scoring function không dùng LLM (keyword matching hoặc regex).

### Exercise 4 — Production System Prompt
Viết complete system prompt cho một trong các agents sau:
- SQL Query Agent (nhận question in plain English → trả về SQL query)
- PR Reviewer Agent (review GitHub PR description → checklist)
- Meeting Summarizer Agent (transcript → action items + decisions)

### Exercise 5 — Meta-Prompting Pipeline
Build một pipeline: User describes task → Meta-prompt generates system prompt → System prompt is evaluated → Best prompt is deployed. Chain tất cả lại.


In [ ]:
# === EXERCISE STARTER CODE ===

# Exercise 1: Intent Classifier
INTENTS = ["REFUND", "SHIPPING", "TECHNICAL", "BILLING", "OTHER"]

# TODO: Add your few-shot examples here
few_shot_intent_examples = [
    # {"message": "...", "intent": "..."}
]

def classify_intent(message: str) -> str:
    # TODO: Build few-shot prompt and call API
    pass

# Test messages
test_messages = [
    "I want my money back, the product was broken",
    "Where is my order? It's been 2 weeks",
    "The app keeps crashing on iOS 17",
    "You charged me twice this month",
    "I just wanted to say thank you!"
]

# for msg in test_messages:
#     print(f"{classify_intent(msg):12} | {msg}")

print("💪 Exercise starters ready! Implement the TODO sections above.")